In [27]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
import seaborn as sns
import joblib

In [3]:
# --- 1. CHARGEMENT ET PRÉPARATION DES DONNÉES ---
print(">>> Chargement des données...")
df = pd.read_csv('dataset/wind-turbine-predictive-maintenance/wind_turbine_maintenance_data.csv')

# Création d'un index temporel simulé pour l'exercice
dates = pd.date_range(start='2022-01-01', periods=len(df)//2, freq='h')
df.loc[df['Turbine_ID'] == 1, 'Timestamp'] = dates
df.loc[df['Turbine_ID'] == 2, 'Timestamp'] = dates
df = df.sort_values(by=['Turbine_ID', 'Timestamp'])

>>> Chargement des données...


In [4]:
# --- 2. FEATURE ENGINEERING (Pour les modèles avancés) ---
print(">>> Génération des Features Temporelles...")
df_eng = df.copy()
cols_cibles = ['Vibration_Level_mmps', 'Gearbox_Oil_Temp_C', 'Generator_Bearing_Temp_C']

for col in cols_cibles:
    # Lags (Passé immédiat)
    df_eng[f'{col}_Lag1'] = df_eng.groupby('Turbine_ID')[col].shift(1)
    df_eng[f'{col}_Lag2'] = df_eng.groupby('Turbine_ID')[col].shift(2)
    # Rolling Stats (Tendances et Volatilité)
    df_eng[f'{col}_Mean_6h'] = df_eng.groupby('Turbine_ID')[col].transform(lambda x: x.rolling(6).mean())
    df_eng[f'{col}_Std_6h'] = df_eng.groupby('Turbine_ID')[col].transform(lambda x: x.rolling(6).std())
    df_eng[f'{col}_Mean_24h'] = df_eng.groupby('Turbine_ID')[col].transform(lambda x: x.rolling(24).mean())

# Nettoyage des NaN
df_eng = df_eng.dropna().reset_index(drop=True)

# Pour la baseline, on prend les mêmes lignes (alignement)
df_raw = df.loc[df_eng.index].reset_index(drop=True)

>>> Génération des Features Temporelles...


In [5]:
df_eng.head()

,Turbine_ID,Rotor_Speed_RPM,Wind_Speed_mps,Power_Output_kW,Gearbox_Oil_Temp_C,Generator_Bearing_Temp_C,Vibration_Level_mmps,Ambient_Temp_C,Humidity_pct,Maintenance_Label,...,Gearbox_Oil_Temp_C_Lag1,Gearbox_Oil_Temp_C_Lag2,Gearbox_Oil_Temp_C_Mean_6h,Gearbox_Oil_Temp_C_Std_6h,Gearbox_Oil_Temp_C_Mean_24h,Generator_Bearing_Temp_C_Lag1,Generator_Bearing_Temp_C_Lag2,Generator_Bearing_Temp_C_Mean_6h,Generator_Bearing_Temp_C_Std_6h,Generator_Bearing_Temp_C_Mean_24h
0,1,14.153206,6.485153,1455.348505,69.281994,76.070469,1.750852,10.865905,63.853174,0,...,63.255494,65.468827,64.670908,2.699740,64.504565,78.030647,67.967974,73.206065,4.798771,74.928410
1,1,13.852280,9.389603,1414.475948,59.864691,79.414188,1.910139,12.093197,52.718208,1,...,69.281994,63.255494,64.003958,3.353401,63.973296,76.070469,78.030647,73.397556,5.057006,75.161116
2,1,15.905122,7.396096,1530.444912,66.286037,75.119659,2.174383,17.186782,60.073064,0,...,59.864691,69.281994,64.231394,3.472241,64.116004,79.414188,76.070469,74.525894,4.420673,75.276897
3,1,14.834690,7.312825,1506.835125,62.952956,84.447738,1.864839,5.432937,55.449668,0,...,66.286037,59.864691,64.518333,3.237970,64.138093,75.119659,79.414188,76.841779,5.444012,75.795606
4,1,15.686260,6.387284,1452.806813,70.444753,75.321400,1.784451,6.423481,66.795977,2,...,62.952956,66.286037,65.347654,4.062379,64.773223,84.447738,75.119659,78.067350,3.542353,76.085703


In [6]:
df_raw

,Turbine_ID,Rotor_Speed_RPM,Wind_Speed_mps,Power_Output_kW,Gearbox_Oil_Temp_C,Generator_Bearing_Temp_C,Vibration_Level_mmps,Ambient_Temp_C,Humidity_pct,Maintenance_Label,Timestamp
0,1,15.496714,7.861736,1564.768854,72.615149,73.829233,1.953173,17.896064,67.674347,0,2022-01-01 00:00:00
1,1,15.079832,7.840483,1502.222183,62.861035,72.340913,1.976505,11.110395,52.320235,0,2022-01-01 01:00:00
2,1,17.559488,8.394233,1512.221917,62.422822,71.998731,2.189488,11.455170,53.644403,0,2022-01-01 02:00:00
3,1,15.822545,6.779156,1520.886360,55.201649,68.359070,2.039372,13.692333,61.713683,0,2022-01-01 03:00:00
4,1,15.564464,8.025501,1547.319325,68.295953,86.703732,2.214197,10.482082,64.191021,1,2022-01-01 04:00:00
...,...,...,...,...,...,...,...,...,...,...,...
34989,2,15.025206,6.002887,1527.455727,58.474194,65.232194,2.354907,0.652312,59.498684,0,2023-12-29 21:00:00
34990,2,13.815059,7.392016,1599.083862,64.879021,79.435889,2.062248,11.440910,71.304337,0,2023-12-29 22:00:00
34991,2,13.859661,7.255955,1448.279303,68.503093,90.479562,1.625516,9.259278,47.620834,1,2023-12-29 23:00:00
34992,2,16.012387,8.315283,1398.786398,56.191956,70.615420,2.031727,6.267608,55.464952,0,2023-12-30 00:00:00


In [7]:
# --- 3. DÉFINITION DU SPLIT CHRONOLOGIQUE ---
split_idx = int(len(df_eng) * 0.80)

# Jeux de données "pour rendre plus Intelligents" (Avec features)
X_train_eng = df_eng.iloc[:split_idx].drop(['Turbine_ID', 'Timestamp', 'Maintenance_Label'], axis=1)
y_train = df_eng.iloc[:split_idx]['Maintenance_Label']
X_test_eng = df_eng.iloc[split_idx:].drop(['Turbine_ID', 'Timestamp', 'Maintenance_Label'], axis=1)
y_test = df_eng.iloc[split_idx:]['Maintenance_Label']

# Jeux de données "Bruts" (Pour la baseline)
raw_features = ['Rotor_Speed_RPM', 'Wind_Speed_mps', 'Power_Output_kW', 'Gearbox_Oil_Temp_C', 
                'Generator_Bearing_Temp_C', 'Vibration_Level_mmps', 'Ambient_Temp_C', 'Humidity_pct']
X_train_raw = df_raw.iloc[:split_idx][raw_features]
X_test_raw = df_raw.iloc[split_idx:][raw_features]

In [8]:
X_test_eng.head()

,Rotor_Speed_RPM,Wind_Speed_mps,Power_Output_kW,Gearbox_Oil_Temp_C,Generator_Bearing_Temp_C,Vibration_Level_mmps,Ambient_Temp_C,Humidity_pct,Vibration_Level_mmps_Lag1,Vibration_Level_mmps_Lag2,...,Gearbox_Oil_Temp_C_Lag1,Gearbox_Oil_Temp_C_Lag2,Gearbox_Oil_Temp_C_Mean_6h,Gearbox_Oil_Temp_C_Std_6h,Gearbox_Oil_Temp_C_Mean_24h,Generator_Bearing_Temp_C_Lag1,Generator_Bearing_Temp_C_Lag2,Generator_Bearing_Temp_C_Mean_6h,Generator_Bearing_Temp_C_Std_6h,Generator_Bearing_Temp_C_Mean_24h
27995,15.423383,5.850375,1476.647758,69.004368,74.539076,1.580128,4.820281,66.653445,2.200385,1.917654,...,57.000324,66.740246,65.728410,4.397322,64.288044,80.383820,68.890659,74.948781,5.071903,73.649973
27996,17.345487,7.385393,1671.523501,65.810647,79.035235,2.107440,10.352638,57.040269,1.580128,2.200385,...,69.004368,57.000324,65.650166,4.389712,64.538114,74.539076,80.383820,76.026053,5.151816,73.573069
27997,17.145271,8.653591,1528.750000,70.531438,74.704917,1.864423,5.009154,57.152518,2.107440,1.580128,...,65.810647,69.004368,66.237305,4.822052,64.652088,79.035235,74.539076,76.540787,4.762724,73.621877
27998,15.352294,8.040059,1380.285552,63.373419,73.739746,2.226287,12.858733,40.315660,1.864423,2.107440,...,70.531438,65.810647,65.410074,4.815579,64.576277,74.704917,79.035235,75.215576,4.103696,73.260349
27999,14.062970,6.519334,1504.815188,64.320880,75.181127,2.041793,11.293785,53.151013,2.226287,1.864423,...,63.373419,70.531438,65.006846,4.783104,64.472873,73.739746,74.704917,76.263987,2.742380,73.695052


In [9]:
plt.figure(figsize=(16, 14))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', 
            square=True, linewidths=1, cbar_kws={'shrink': 0.8})
plt.title('Matrice de Corrélation', fontsize=25, fontweight='bold')
plt.tight_layout()
plt.show()

NameError: name 'correlation_matrix' is not defined

<Figure size 1600x1400 with 0 Axes>

In [20]:
# --- 4. LE TOURNOI DES MODÈLES ---

# A) BASELINE (Reg Log sur données brutes)
print("\n>>> 1. Entraînement Baseline (LogReg Simple)...")
model_baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(class_weight='balanced', random_state=42))
])
model_baseline.fit(X_train_raw, y_train)
f1_baseline = f1_score(y_test, model_baseline.predict(X_test_raw), average='weighted')

# B) CHALLENGER (Lasso sur Features Temporelles)
print(">>> 2. Entraînement Challenger (Lasso + Features Temporelles)...")
model_lasso = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(
        penalty='l1', 
        solver='saga',    
        C=0.5, 
        # multi_class='auto',
        class_weight='balanced', 
        random_state=42,
        max_iter=5000     
    ))
])
model_lasso.fit(X_train_eng, y_train)
f1_lasso = f1_score(y_test, model_lasso.predict(X_test_eng), average='weighted')

# C) CHAMPION (Random Forest sur Features Temporelles)
print(">>> 3. Entraînement Champion (Random Forest + Features Temporelles)...")
model_champion = RandomForestClassifier(
    n_estimators=200, 
    max_depth=15,
    min_samples_leaf=2,
    min_samples_split=8,
    class_weight='balanced', 
    random_state=42, 
    n_jobs=-1)
model_champion.fit(X_train_eng, y_train)
f1_champion = f1_score(y_test, model_champion.predict(X_test_eng), average='weighted')


>>> 1. Entraînement Baseline (LogReg Simple)...
>>> 2. Entraînement Challenger (Lasso + Features Temporelles)...


c:\Users\Juju\anaconda3\envs\projet-final\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Juju\anaconda3\envs\projet-final\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


>>> 3. Entraînement Champion (Random Forest + Features Temporelles)...


In [ ]:
f1_champion

0.7605586356524867

In [21]:
# --- 5. RÉSULTATS COMPARATIFS ---
print("\n" + "="*45)
print("   🏆 ÉVOLUTION DU PROJET (F1-SCORE WEIGHTED)")
print("="*45)
print(f"1. BASELINE (Naïve)       : {f1_baseline:.2%}")
print(f"   -> Approche : Données brutes, Modèle simple")
print("-" * 45)
print(f"2. CHALLENGER (Ingénierie): {f1_lasso:.2%}")
print(f"   -> Approche : Features Temporelles, Lasso")
print(f"   -> Gain vs Baseline : {f1_lasso - f1_baseline:+.2%}")
print("-" * 45)
print(f"3. CHAMPION (Non-Linéaire): {f1_champion:.2%}")
print(f"   -> Approche : Random Forest, Interactions")
print(f"   -> Gain vs Challenger : {f1_champion - f1_lasso:+.2%}")
print("="*45)

# --- 6. SAUVEGARDE DU VAINQUEUR ---
# Car le Random Forest gérera mieux les cas bizarres (non-linéaires) non vus dans le test.
print(f"\n>>> Sauvegarde du modèle CHAMPION (Random Forest)...")
joblib.dump(model_champion, 'gtc_model_advanced.pkl') 
print("✅ Modèle sauvegardé sous 'gtc_model_advanced.pkl'. Prêt pour app.py !")


   🏆 ÉVOLUTION DU PROJET (F1-SCORE WEIGHTED)
1. BASELINE (Naïve)       : 42.62%
   -> Approche : Données brutes, Modèle simple
---------------------------------------------
2. CHALLENGER (Ingénierie): 34.19%
   -> Approche : Features Temporelles, Lasso
   -> Gain vs Baseline : -8.43%
---------------------------------------------
3. CHAMPION (Non-Linéaire): 76.18%
   -> Approche : Random Forest, Interactions
   -> Gain vs Challenger : +42.00%

>>> Sauvegarde du modèle CHAMPION (Random Forest)...
✅ Modèle sauvegardé sous 'gtc_model_advanced.pkl'. Prêt pour app.py !


In [24]:
print("\n>>> 6. OPTIMIZATION : Grid Search Random Forest (Features Temporelles)...")

random_forest = RandomForestClassifier(
    class_weight='balanced',  
    random_state=42,
    n_jobs=-1
)

params = {
    'max_depth': [13, 15, 17],  
    'min_samples_leaf': [1, 2, 5, 8],
    'min_samples_split': [2, 3, 4],
    'n_estimators': [200, 250, 300]
}

# ✅ CORRECTIONS ESSENTIELLES
gridsearch = GridSearchCV(
    random_forest,
    param_grid=params,
    cv=3,
    verbose=1,
    scoring='f1_weighted',              
    n_jobs=-1                           
)

print("Lancement du grid search...")
gridsearch.fit(X_train_eng, y_train)    
print("...Done.\n")

print("Best hyperparameters:", gridsearch.best_params_)
print(f"Best CV F1-score: {gridsearch.best_score_:.4f}")

# Évaluation finale
f1_gridsearch = f1_score(y_test, gridsearch.predict(X_test_eng), average='weighted')

print("\n" + "="*50)
print("   🏆 ÉVOLUTION DU PROJET (F1-SCORE WEIGHTED)")
print("="*50)
print(f"1. BASELINE (LogReg)         : {f1_baseline:.4f}")
print(f"2. CHALLENGER (Lasso)        : {f1_lasso:.4f}")
print(f"3. CHAMPION (RF hardcoded)   : {f1_champion:.4f}")
print(f"4. OPTIMIZED (RF GridSearch) : {f1_gridsearch:.4f} ⭐")
print("="*50)


>>> 6. OPTIMIZATION : Grid Search Random Forest (Features Temporelles)...
Lancement du grid search...
Fitting 3 folds for each of 108 candidates, totalling 324 fits
...Done.

Best hyperparameters: {'max_depth': 13, 'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 300}
Best CV F1-score: 0.7588

   🏆 ÉVOLUTION DU PROJET (F1-SCORE WEIGHTED)
1. BASELINE (LogReg)         : 0.4262
2. CHALLENGER (Lasso)        : 0.3419
3. CHAMPION (RF hardcoded)   : 0.7618
4. OPTIMIZED (RF GridSearch) : 0.7609 ⭐


In [29]:
print("\n>>> 7. OPTIMIZATION : Grid Search Logistic Regression (Features Temporelles)...")

# ✅ LogReg avec les mêmes fondations que ton baseline
logreg = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=5000,
    solver='saga',
    n_jobs=-1
)

# ✅ Hyperparamètres pertinents pour LogReg
params = {
    'C': [0.1, 0.5, 1, 5, 10],  # Force de régularisation (inverse)
    'l1_ratio': [0, 0.5, 1]              # Compatible avec l1 et l2
}

# ✅ GridSearchCV avec les mêmes standards
gridsearch_logreg = GridSearchCV(
    logreg,
    param_grid=params,
    cv=3,
    verbose=1,
    scoring='f1_weighted',
    n_jobs=-1
)

print("Lancement du grid search LogReg...")
gridsearch_logreg.fit(X_train_eng, y_train)
print("...Done.\n")

print("Best hyperparameters:", gridsearch_logreg.best_params_)
print(f"Best CV F1-score: {gridsearch_logreg.best_score_:.4f}")

# Évaluation finale
f1_gridsearch_logreg = f1_score(y_test, gridsearch_logreg.predict(X_test_eng), average='weighted')

print("\n" + "="*50)
print("   🏆 ÉVOLUTION DU PROJET (F1-SCORE WEIGHTED)")
print("="*50)
print(f"1. BASELINE (LogReg)            : {f1_baseline:.4f}")
print(f"2. CHALLENGER (Lasso)           : {f1_lasso:.4f}")
print(f"3. CHAMPION (RF hardcoded)      : {f1_champion:.4f}")
print(f"4. OPTIMIZED (RF GridSearch)    : {f1_gridsearch:.4f}")
print(f"5. OPTIMIZED (LogReg GridSearch): {f1_gridsearch_logreg:.4f} ⭐")
print("="*50)


>>> 7. OPTIMIZATION : Grid Search Logistic Regression (Features Temporelles)...
Lancement du grid search LogReg...
Fitting 3 folds for each of 15 candidates, totalling 45 fits


c:\Users\Juju\anaconda3\envs\projet-final\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


...Done.

Best hyperparameters: {'C': 10, 'l1_ratio': 1}
Best CV F1-score: 0.7538

   🏆 ÉVOLUTION DU PROJET (F1-SCORE WEIGHTED)
1. BASELINE (LogReg)            : 0.4262
2. CHALLENGER (Lasso)           : 0.3419
3. CHAMPION (RF hardcoded)      : 0.7618
4. OPTIMIZED (RF GridSearch)    : 0.7609
5. OPTIMIZED (LogReg GridSearch): 0.5687 ⭐


c:\Users\Juju\anaconda3\envs\projet-final\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [28]:
print("\n>>> 8. OPTIMIZATION : Grid Search XGBoost (Features Temporelles)...")

# ✅ XGBoost avec les mêmes fondations que tes autres modèles
xgb_model = XGBClassifier(
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',  # Pour classification multiclass
    verbosity=0  # Réduit le bruit de logs
)

# ✅ Hyperparamètres pertinents pour XGBoost
# Stratégie : d'abord les params importants, puis affiner
params = {
    'max_depth': [8, 10, 15],              # Profondeur des arbres
    'learning_rate': [0.01, 0.05, 0.1],    # Shrinkage (learning rate)
    'n_estimators': [100, 150, 200],   # Nombre d'arbres
    'min_child_weight': [1, 3, 5],         # Min samples per leaf équivalent
    'subsample': [0.7, 0.9, 1.0],    # Fraction des samples par arbre
    'colsample_bytree': [0.7, 0.8, 0.9],  # Fraction des features par arbre
}

# ✅ GridSearchCV avec les mêmes standards
gridsearch_xgb = GridSearchCV(
    xgb_model,
    param_grid=params,
    cv=3,
    verbose=1,
    scoring='f1_weighted',
    n_jobs=-1
)

print("Lancement du grid search XGBoost...")
print(f"Nombre total de combinaisons : {4*3*4*3*4*3} × 3 folds = 1728 fits (⚠️ long !)")
gridsearch_xgb.fit(X_train_eng, y_train)
print("...Done.\n")

print("Best hyperparameters:", gridsearch_xgb.best_params_)
print(f"Best CV F1-score: {gridsearch_xgb.best_score_:.4f}")

# Évaluation finale
f1_gridsearch_xgb = f1_score(
    y_test, 
    gridsearch_xgb.predict(X_test_eng), 
    average='weighted'
)

print("\n" + "="*60)
print("   🏆 ÉVOLUTION DU PROJET (F1-SCORE WEIGHTED)")
print("="*60)
print(f"1. BASELINE (LogReg)              : {f1_baseline:.4f}")
print(f"2. CHALLENGER (Lasso)             : {f1_lasso:.4f}")
print(f"3. CHAMPION (RF hardcoded)        : {f1_champion:.4f}")
print(f"4. OPTIMIZED (RF GridSearch)      : {f1_gridsearch:.4f}")
print(f"5. OPTIMIZED (LogReg GridSearch)  : {f1_gridsearch_logreg:.4f}")
print(f"6. OPTIMIZED (XGBoost GridSearch) : {f1_gridsearch_xgb:.4f} ⭐")
print("="*60)


>>> 8. OPTIMIZATION : Grid Search XGBoost (Features Temporelles)...
Lancement du grid search XGBoost...
Nombre total de combinaisons : 1728 × 3 folds = 1728 fits (⚠️ long !)
Fitting 3 folds for each of 729 candidates, totalling 2187 fits
...Done.

Best hyperparameters: {'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 15, 'min_child_weight': 3, 'n_estimators': 150, 'subsample': 0.7}
Best CV F1-score: 0.7588

   🏆 ÉVOLUTION DU PROJET (F1-SCORE WEIGHTED)
1. BASELINE (LogReg)              : 0.4262
2. CHALLENGER (Lasso)             : 0.3419
3. CHAMPION (RF hardcoded)        : 0.7618
4. OPTIMIZED (RF GridSearch)      : 0.7609
5. OPTIMIZED (LogReg GridSearch)  : 0.3495
6. OPTIMIZED (XGBoost GridSearch) : 0.7615 ⭐
